# 23 - K-Means: clustering, elbow y silhouette con flujo completo

## Objetivos de aprendizaje

En esta sesion aprenderas a:

1. Entender que resuelve K-Means dentro de ciencia de datos.
2. Conocer el algoritmo paso a paso.
3. Interpretar formulas clave como inercia.
4. Aplicar un flujo con `train_test_split` en clustering.
5. Usar e interpretar las metricas de elbow y silhouette.
6. Reconocer riesgos y malas practicas comunes.


## Ruta de la sesion

1. Recordatorio breve de ciencia de datos.
2. Que es K-Means y cuando usarlo.
3. Formulas clave (objetivo, centroides, inercia).
4. Por que la curva elbow baja cuando aumenta `k`.
5. Que mide silhouette.
6. Flujo de extremo a extremo con datos de segmentacion sintetica.
7. Visualizaciones de apoyo.
8. Puntos clave y puntos de cuidado.
9. Ejercicios para pensar.


## 1) Recordatorio rapido: ciencia de datos y clustering

Ciencia de datos busca apoyar decisiones con evidencia cuantitativa.

En problemas **supervisados** tienes una variable objetivo (ej. clase o precio).
En problemas **no supervisados** no hay etiqueta; quieres encontrar estructura oculta.

K-Means es un metodo no supervisado para agrupar observaciones parecidas.


## 2) Que es K-Means

K-Means intenta dividir los datos en `k` grupos.
Cada grupo se representa por su centroide (promedio de puntos del grupo).

Idea intuitiva:

1. Propone `k` centroides iniciales.
2. Asigna cada punto al centroide mas cercano.
3. Recalcula centroides como promedio de puntos asignados.
4. Repite 2-3 hasta converger.


## 3) Formulas clave

### 3.1 Funcion objetivo de K-Means

$$\min_{C_1,\dots,C_k,\mu_1,\dots,\mu_k} \sum_{r=1}^{k} \sum_{x_i \in C_r} ||x_i - \mu_r||_2^2$$

donde:

- $C_r$ es el conjunto de puntos en el cluster $r$.
- $\mu_r$ es el centroide del cluster $r$.

### 3.2 Actualizacion del centroide

$$\mu_r = \frac{1}{|C_r|} \sum_{x_i \in C_r} x_i$$

### 3.3 Inercia (WCSS/SSE)

En `scikit-learn`, `inertia_` representa:

$$\text{Inercia} = \sum_{i=1}^{n} \min_{r \in \{1,\dots,k\}} ||x_i - \mu_r||_2^2$$

Menor inercia implica clusters mas compactos respecto a sus centroides.


## 4) Aplicaciones interesantes de K-Means

1. Segmentacion de clientes (marketing y ofertas).
2. Agrupacion de zonas geograficas por comportamiento.
3. Agrupar documentos por temas aproximados.
4. Compresion de imagen por cuantizacion de colores.
5. Deteccion preliminar de patrones atipicos (clusters pequenos o lejanos).

Hoy simularemos un caso de segmentacion de clientes con 2 variables de negocio.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_samples, silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


## 5) Datos de ejemplo: segmentacion sintetica

Creamos dos variables con significado de negocio:

- `frecuencia_mensual`: numero de compras al mes.
- `ticket_promedio_usd`: gasto promedio por compra.

Las escalas son distintas a proposito para mostrar por que normalizar importa.


In [ ]:
X_latente, _ = make_blobs(
    n_samples=850,
    centers=4,
    cluster_std=[0.85, 1.0, 0.9, 1.1],
    n_features=2,
    random_state=42,
)

frecuencia = 8 + 2.4 * X_latente[:, 0]           # escala ~ decenas
ticket = 320 + 85 * X_latente[:, 1]              # escala ~ cientos

X = np.column_stack([frecuencia, ticket])
feature_names = ["frecuencia_mensual", "ticket_promedio_usd"]

print("Forma de X:", X.shape)
print("Rangos aproximados:")
print(f"{feature_names[0]}: {X[:,0].min():.2f} a {X[:,0].max():.2f}")
print(f"{feature_names[1]}: {X[:,1].min():.2f} a {X[:,1].max():.2f}")


In [ ]:
plt.figure(figsize=(7.5, 5))
plt.scatter(X[:, 0], X[:, 1], s=20, alpha=0.7)
plt.xlabel(feature_names[0])
plt.ylabel(feature_names[1])
plt.title("Datos sinteticos para segmentacion")
plt.grid(alpha=0.2)
plt.show()


## 6) Flujo con train/test split en clustering

En clustering no siempre se hace train/test como en clasificacion.
Aun asi, puede ser util para:

1. elegir hiperparametros sin usar todo el dataset,
2. evaluar estabilidad de estructura en datos no vistos,
3. evitar fuga de informacion al escalar (fit scaler solo en train).


In [ ]:
X_train, X_test = train_test_split(X, test_size=0.25, random_state=2026)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print("Train:", X_train.shape, "Test:", X_test.shape)


## 7) Impacto de normalizar: comparacion rapida

Comparamos `k=4` con y sin normalizar.
Usamos silhouette para comparar separacion/cohesion bajo cada geometria.


In [ ]:
k_demo = 4

kmeans_raw = KMeans(n_clusters=k_demo, random_state=42, n_init=20)
labels_raw_test = kmeans_raw.fit(X_train).predict(X_test)
sil_raw_test = silhouette_score(X_test, labels_raw_test)

kmeans_sc = KMeans(n_clusters=k_demo, random_state=42, n_init=20)
labels_sc_test = kmeans_sc.fit(X_train_sc).predict(X_test_sc)
sil_sc_test = silhouette_score(X_test_sc, labels_sc_test)

print(f"Silhouette test sin normalizar (k=4): {sil_raw_test:.3f}")
print(f"Silhouette test con normalizar (k=4): {sil_sc_test:.3f}")


## 8) Elbow y Silhouette para elegir `k`

### 8.1 Elbow (codo)

Elbow observa la inercia al variar `k`.
Buscas un punto donde agregar mas clusters ya mejora poco.

### 8.2 Silhouette

Para cada punto `i`:

- $a(i)$ = distancia promedio a puntos de su propio cluster.
- $b(i)$ = menor distancia promedio a otro cluster (el mas cercano alternativo).

$$s(i) = \frac{b(i) - a(i)}{\max\{a(i), b(i)\}}$$

Rango de silhouette: `[-1, 1]`.

- cercano a 1: bien ubicado,
- cerca de 0: en frontera,
- negativo: posible mala asignacion.


In [ ]:
def test_inertia(kmeans_model, X_eval):
    # Distancias euclidianas a cada centroide para cada punto.
    dists = kmeans_model.transform(X_eval)
    # Inercia equivalente: suma de distancia al centroide asignado al cuadrado.
    return np.sum(np.min(dists ** 2, axis=1))


def safe_silhouette(X_eval, labels):
    unique_labels = np.unique(labels)
    if len(unique_labels) < 2:
        return np.nan
    return silhouette_score(X_eval, labels)


In [ ]:
k_values = list(range(2, 11))

inertia_train = []
inertia_test = []
sil_train = []
sil_test = []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=30)
    km.fit(X_train_sc)

    inertia_train.append(km.inertia_)
    inertia_test.append(test_inertia(km, X_test_sc))

    labels_train = km.labels_
    labels_test = km.predict(X_test_sc)

    sil_train.append(safe_silhouette(X_train_sc, labels_train))
    sil_test.append(safe_silhouette(X_test_sc, labels_test))

inertia_train = np.array(inertia_train)
inertia_test = np.array(inertia_test)
sil_train = np.array(sil_train)
sil_test = np.array(sil_test)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

axes[0].plot(k_values, inertia_train, marker="o", label="train")
axes[0].plot(k_values, inertia_test, marker="s", label="test")
axes[0].set_title("Curva Elbow (inercia)")
axes[0].set_xlabel("Numero de clusters (k)")
axes[0].set_ylabel("Inercia")
axes[0].set_xticks(k_values)
axes[0].grid(alpha=0.25)
axes[0].legend()

axes[1].plot(k_values, sil_train, marker="o", label="train")
axes[1].plot(k_values, sil_test, marker="s", label="test")
axes[1].set_title("Silhouette promedio")
axes[1].set_xlabel("Numero de clusters (k)")
axes[1].set_ylabel("Silhouette")
axes[1].set_xticks(k_values)
axes[1].grid(alpha=0.25)
axes[1].legend()

plt.tight_layout()
plt.show()


## 9) Por que elbow baja cuando `k` aumenta

La inercia es una suma de distancias cuadradas al centroide mas cercano.
Al aumentar `k`, das mas centroides disponibles para aproximar los datos.

Formalmente, el problema con `k+1` clusters contiene al de `k` como caso particular.
Por eso el minimo alcanzable de inercia con `k+1` **no puede ser mayor** que con `k`.

Conclusion: la curva elbow es monotona no creciente.
Lo dificil no es que baje, sino decidir en que punto deja de bajar "lo suficiente".


In [ ]:
best_k_by_sil = int(k_values[np.nanargmax(sil_test)])
print(f"k sugerido por silhouette(test): {best_k_by_sil}")

k_final = best_k_by_sil
kmeans_final = KMeans(n_clusters=k_final, random_state=42, n_init=50)
kmeans_final.fit(X_train_sc)

labels_train_final = kmeans_final.labels_
labels_test_final = kmeans_final.predict(X_test_sc)

centroides_original = scaler.inverse_transform(kmeans_final.cluster_centers_)
print("Centroides finales (escala original):")
print(centroides_original)


In [ ]:
plt.figure(figsize=(7.8, 5.2))
plt.scatter(
    X_train[:, 0],
    X_train[:, 1],
    c=labels_train_final,
    cmap="tab10",
    s=26,
    alpha=0.78,
    label="train",
)
plt.scatter(
    centroides_original[:, 0],
    centroides_original[:, 1],
    marker="X",
    s=260,
    c="black",
    label="centroides",
)

plt.xlabel(feature_names[0])
plt.ylabel(feature_names[1])
plt.title(f"Clusters train con K-Means (k={k_final})")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


In [ ]:
sil_values = silhouette_samples(X_train_sc, labels_train_final)

plt.figure(figsize=(8.2, 5.5))
y_lower = 10
for cluster_id in range(k_final):
    vals = np.sort(sil_values[labels_train_final == cluster_id])
    size = vals.shape[0]
    y_upper = y_lower + size
    plt.fill_betweenx(np.arange(y_lower, y_upper), 0, vals, alpha=0.7)
    plt.text(-0.05, y_lower + 0.5 * size, str(cluster_id))
    y_lower = y_upper + 10

sil_avg = np.mean(sil_values)
plt.axvline(sil_avg, color="red", linestyle="--", label=f"media={sil_avg:.3f}")
plt.title(f"Silhouette plot (train) para k={k_final}")
plt.xlabel("Coeficiente silhouette")
plt.ylabel("Clusters")
plt.legend()
plt.grid(alpha=0.15)
plt.show()


## 10) Puntos clave y puntos de cuidado

### Puntos clave

1. Normalizar variables suele ser obligatorio en K-Means.
2. Elbow y silhouette son guias, no reglas absolutas.
3. `random_state` y `n_init` importan para estabilidad.

### Puntos de cuidado

1. K-Means asume clusters aproximadamente esfericos y de tamano comparable.
2. Es sensible a outliers (mueven centroides).
3. Si hay estructura no convexa, puede fallar (ej. formas de media luna).
4. Menor inercia no siempre implica mejor interpretacion de negocio.
5. Elegir `k` solo por grafica sin contexto puede llevar a decisiones malas.


## 11) Ejercicios de pensamiento (no copiar/pegar)

Primero escribe tu argumento en texto. Luego valida con codigo.


### Ejercicio 1: elbow vs silhouette en conflicto

Supongamos que elbow sugiere `k=4`, pero silhouette maxima ocurre en `k=3`.

1. Que decision tomarias y por que.
2. Que informacion de negocio pedirias para decidir.
3. Que experimento adicional correria tu equipo.


In [ ]:
# Escribe aqui evidencia adicional para decidir entre k=3 y k=4.


### Ejercicio 2: sensibilidad a outliers

Agrega manualmente 8 puntos extremos al dataset y repite el analisis.

1. Como cambian centroides.
2. Como cambian elbow y silhouette.
3. Que estrategia propones para robustecer el flujo.


In [ ]:
# Implementa aqui el experimento de outliers.


### Ejercicio 3: impacto de no normalizar

Disena una prueba para mostrar como domina la variable de mayor escala.

1. Multiplica `ticket_promedio_usd` por 10.
2. Repite clustering sin escalar y con escalar.
3. Explica por que cambian (o no) los clusters.


In [ ]:
# Implementa aqui el experimento de escalas.


### Ejercicio 4: estabilidad de K-Means

Cambia `random_state` y `n_init`.

1. Cuanto se mueven los centroides entre corridas.
2. En que rango varia silhouette.
3. Que configuracion minima de `n_init` recomendarias y por que.


In [ ]:
# Implementa aqui comparacion de estabilidad entre corridas.


### Ejercicio 5: traduccion a decision de negocio

Redacta una recomendacion para un equipo comercial:

1. cuantos segmentos usar,
2. como describir cada segmento en lenguaje no tecnico,
3. que accion concreta haria el equipo con cada segmento,
4. que riesgo existe si la distribucion de clientes cambia en 6 meses.


In [ ]:
# Escribe aqui tu propuesta ejecutiva.


## 12) Cierre

Ideas clave:

1. K-Means minimiza distancias cuadraticas a centroides.
2. La inercia siempre baja al subir `k`, por construccion del problema.
3. Silhouette complementa elbow al medir separacion/cohesion relativa.
4. Normalizar y controlar inicializacion son practicas criticas.
5. Elegir `k` requiere evidencia tecnica + criterio de negocio.
